# Langevin to Stationary Density

Chapter 21: Fokker-Planck and Distribution Dynamics · *Mathematics for AI and Machine Learning* by Fuheng Wu

Simulate many particles and compare terminal histogram with Boltzmann stationary density.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-21/langevin-stationary/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

torch.manual_seed(42)

# The demo's defaults: dX = -U'(X) dt + sqrt(2 D) dW, U(x) = (x^2 - 1)^2 / 4, dt = 0.01
D, steps, N, dt = 0.45, 2200, 500, 0.01


def grad_U(x):
    return x * (x ** 2 - 1)


x = torch.zeros(20000, dtype=torch.float64)          # many more particles than the demo's 500
for _ in range(steps):
    x = x - grad_U(x) * dt + math.sqrt(2 * D * dt) * torch.randn_like(x)

# ── Compare with the Boltzmann density p ∝ exp(-U / D) ─────────────────────
grid = torch.linspace(-3, 3, 6001, dtype=torch.float64)
boltz = torch.exp(-0.25 * (grid ** 2 - 1) ** 2 / D)
boltz = boltz / torch.trapezoid(boltz, grid)
edges = torch.linspace(-2.0, 2.0, 9, dtype=torch.float64)
counts = torch.histc(x, bins=8, min=-2.0, max=2.0) / len(x)
print("   bin            particles   Boltzmann")
for i in range(8):
    sel = (grid >= edges[i]) & (grid < edges[i + 1])
    print(f"[{edges[i].item():+.1f}, {edges[i + 1].item():+.1f})     {counts[i].item():.4f}      {torch.trapezoid(boltz[sel], grid[sel]).item():.4f}")

ex2 = torch.trapezoid(grid ** 2 * boltz, grid).item()
print(f"\nE[X^2]: particles {(x ** 2).mean().item():.4f}, Boltzmann {ex2:.4f}")

# Lower temperature D concentrates the mass at the minima x = +-1
for d in (0.1, 0.45, 1.5):
    b = torch.exp(-0.25 * (grid ** 2 - 1) ** 2 / d)
    b = b / torch.trapezoid(b, grid)
    near = (grid.abs() - 1).abs() < 0.3
    print(f"D = {d:4.2f}: Boltzmann mass within 0.3 of a minimum = {torch.trapezoid(b * near, grid).item():.3f}")

   bin            particles   Boltzmann
[-2.0, -1.5)     0.0271      0.0276
[-1.5, -1.0)     0.1646      0.1617
[-1.0, -0.5)     0.1828      0.1814
[-0.5, +0.0)     0.1295      0.1282
[+0.0, +0.5)     0.1261      0.1279
[+0.5, +1.0)     0.1810      0.1810
[+1.0, +1.5)     0.1601      0.1620
[+1.5, +2.0)     0.0282      0.0278

E[X^2]: particles 0.8808, Boltzmann 0.8788
D = 0.10: Boltzmann mass within 0.3 of a minimum = 0.738
D = 0.45: Boltzmann mass within 0.3 of a minimum = 0.459
D = 1.50: Boltzmann mass within 0.3 of a minimum = 0.352
